# Generate layout benchmark cases

This notebook creates reproducible random orders from `data/print_database.json`, runs the genetic baseline one order at a time, saves layouts in `data/training/` with `gen_` names and timestamps, and records completed benchmark data including the UTC completion time. The queue is resumable: each order is removed from `experiments/benchmark_orders.json` only after its layout and result are saved. Use `layout_algorithm_lab.ipynb` to compare algorithms and plot results.

In [15]:
from pathlib import Path
import os
import time
import json
from datetime import datetime, timezone
import numpy as np
import pandas as pd
from IPython.display import HTML, display

ROOT = Path.cwd()
if not (ROOT / 'optimizer').exists():
    ROOT = ROOT.parent
import sys
sys.path.insert(0, str(ROOT))
from experiments.notebooks.layout_lab_utils import (
    benchmark, generate_random_orders, genetic_baseline, load_database, load_json,
)
from optimizer import save_layout
GENETIC_POPULATION = 60
GENETIC_GENERATIONS = 100
MACHINE_WIDTH = 443
database = load_database(ROOT / 'data/print_database.json')
print(f'Database items available: {len(database)}')

Database items available: 1267


In [20]:
# Generate the queue only when it does not exist; rerunning this cell resumes it.
orders_path = ROOT / 'experiments/benchmark_orders.json'
output_dir = ROOT / 'data/training'
results_path = ROOT / 'experiments/benchmark_results.json'

def save_remaining_orders(orders):
    temporary_path = orders_path.with_suffix('.json.tmp')
    try:
        with temporary_path.open('w', encoding='utf-8') as stream:
            json.dump(orders, stream, ensure_ascii=False, indent=4)
            stream.flush()
            os.fsync(stream.fileno())
        os.replace(temporary_path, orders_path)
    finally:
        temporary_path.unlink(missing_ok=True)

if orders_path.exists():
    orders = load_json(orders_path)
    for order in orders:
        order['szerokosc_maszyny'] = MACHINE_WIDTH
    save_remaining_orders(orders)
    print(f'Loaded {len(orders)} remaining orders from experiments/benchmark_orders.json')
if orders_path.exists() and len(orders) == 0:
    orders = generate_random_orders(
        database,
        count=250,
        min_distinct_items=5,
        max_distinct_items=20,
        min_quantity=4,
        max_quantity=15,
        occasional_max_quantity=20,
        high_quantity_probability=0.12,
        widths=(MACHINE_WIDTH,),
        seed=42,
    )
    save_remaining_orders(orders)
    print(f'Saved {len(orders)} orders to experiments/benchmark_orders.json')
orders[:1]

Loaded 0 remaining orders from experiments/benchmark_orders.json
Saved 250 orders to experiments/benchmark_orders.json


[{'case_id': 'random-0001',
  'szerokosc_maszyny': 443,
  'elementy': [{'nazwa': 'STARTUP Wodzisław Śląski 25 cm całość',
    'typ': 'Nadruk/StartUpWodzislawSlaski',
    'w': 253,
    'h': 171,
    'plik': 'C:\\Users\\PC\\PycharmProjects\\app_25\\Pliki_Zrodlowe\\STARTUP Wodzisław Śląski 25 cm całość.cdr'},
   {'nazwa': 'STARTUP Wodzisław Śląski 25 cm całość',
    'typ': 'Nadruk/StartUpWodzislawSlaski',
    'w': 253,
    'h': 171,
    'plik': 'C:\\Users\\PC\\PycharmProjects\\app_25\\Pliki_Zrodlowe\\STARTUP Wodzisław Śląski 25 cm całość.cdr'},
   {'nazwa': 'STARTUP Wodzisław Śląski 25 cm całość',
    'typ': 'Nadruk/StartUpWodzislawSlaski',
    'w': 253,
    'h': 171,
    'plik': 'C:\\Users\\PC\\PycharmProjects\\app_25\\Pliki_Zrodlowe\\STARTUP Wodzisław Śląski 25 cm całość.cdr'},
   {'nazwa': 'STARTUP Wodzisław Śląski 25 cm całość',
    'typ': 'Nadruk/StartUpWodzislawSlaski',
    'w': 253,
    'h': 171,
    'plik': 'C:\\Users\\PC\\PycharmProjects\\app_25\\Pliki_Zrodlowe\\STARTUP Wodzisław

In [ ]:
# Process one queued order at a time. A completed order is removed only after its layout is saved.
# Increase population/generations for stronger results; decrease them for faster experiments.
baseline_rows = load_json(results_path) if results_path.exists() else []
baseline_rows = list(baseline_rows)
while orders:
    order = orders[0]
    results = benchmark(
        order['elementy'],
        order['szerokosc_maszyny'],
        algorithms={'Genetic baseline': lambda items, width: genetic_baseline(items, width, GENETIC_POPULATION, GENETIC_GENERATIONS)},
    )
    baseline = next(row for row in results if row['algorithm'] == 'Genetic baseline')
    output_path = output_dir / f"gen_uklad_ai_{int(time.time())}.json"
    while output_path.exists():
        output_path = output_dir / f"gen_uklad_ai_{int(output_path.stem.rsplit('_', 1)[1]) + 1}.json"
    output_dir.mkdir(parents=True, exist_ok=True)
    temporary_output_path = output_path.with_suffix('.json.tmp')
    save_layout(
        baseline['packer'], baseline['items'], order['szerokosc_maszyny'],
        baseline['height_mm'], temporary_output_path, 'Basic (loose)'
    )
    os.replace(temporary_output_path, output_path)
    baseline_rows.append({
        'case_id': order['case_id'],
        'order': order,
        'layout_path': str(output_path.relative_to(ROOT)),
        'baseline_height_mm': baseline['height_mm'],
        'baseline_utilization_percent': baseline['utilization_percent'],
        'baseline_seconds': baseline['seconds'],
        'generated_at': datetime.now(timezone.utc).isoformat(),
    })
    temporary_results_path = results_path.with_suffix('.json.tmp')
    with temporary_results_path.open('w', encoding='utf-8') as stream:
        json.dump(baseline_rows, stream, ensure_ascii=False, indent=4)
        stream.flush()
        os.fsync(stream.fileno())
    os.replace(temporary_results_path, results_path)
    orders = orders[1:]
    save_remaining_orders(orders)
    print(f"Saved {output_path}; {len(orders)} orders remain")
baseline_df = pd.DataFrame(baseline_rows)
display(baseline_df.style.format({'baseline_height_mm': '{:.2f}', 'baseline_utilization_percent': '{:.2f}', 'baseline_seconds': '{:.3f}'}))
print('Average baseline height:', np.mean(baseline_df['baseline_height_mm']))

Saved c:\Users\PC\my-projects\corel-layout-optimizer\data/training\gen_uklad_ai_1793036044.json; 249 orders remain
Saved c:\Users\PC\my-projects\corel-layout-optimizer\data/training\gen_uklad_ai_1793036060.json; 248 orders remain
Saved c:\Users\PC\my-projects\corel-layout-optimizer\data/training\gen_uklad_ai_1793036124.json; 247 orders remain
Saved c:\Users\PC\my-projects\corel-layout-optimizer\data/training\gen_uklad_ai_1793036135.json; 246 orders remain
Saved c:\Users\PC\my-projects\corel-layout-optimizer\data/training\gen_uklad_ai_1793036140.json; 245 orders remain
Saved c:\Users\PC\my-projects\corel-layout-optimizer\data/training\gen_uklad_ai_1793036182.json; 244 orders remain
Saved c:\Users\PC\my-projects\corel-layout-optimizer\data/training\gen_uklad_ai_1793036187.json; 243 orders remain
Saved c:\Users\PC\my-projects\corel-layout-optimizer\data/training\gen_uklad_ai_1793036195.json; 242 orders remain
Saved c:\Users\PC\my-projects\corel-layout-optimizer\data/training\gen_uklad_ai_

The saved `experiments/benchmark_orders.json` file can be loaded by another notebook or test script. Use the same cases for every new algorithm so the comparison is fair.